
# Free Throw Success Classification (Beginner-Friendly)

This project uses the `combined.csv` dataset (NBA per-game player-season stats from Basketball-Reference) to build **a classification model**.

## Important: What is the target here?
This dataset is **not shot-by-shot free throw attempts** (it does not have a `made/missed` outcome per free throw). Instead it has season-level *rates* like `FT%`.

So we define a clean binary target:

- `target = 1` if a player's **season FT%** is **greater than or equal to the league average FT%** for that same season.
- `target = 0` otherwise.

This gives us a meaningful and well-defined classification task: **is this player an above-average free throw shooter (for that season)?**

We also take care to avoid leakage:
- We **drop `FT%` from the features** (because it's used to build the target).
- We avoid directly using anything that is a trivial re-expression of `FT%`.


# 10. Iteration & Improvement (Train → Evaluate → Improve → Re-train)

This notebook demonstrates a beginner-friendly improvement loop using:
- better preprocessing decisions
- simple feature engineering
- hyperparameter tuning with cross-validation

We will show one example: tuning Logistic Regression and Gradient Boosting.

In [1]:
import joblib
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier

ARTIFACT_DIR = Path('artifacts')
data = joblib.load(ARTIFACT_DIR / 'data_split_and_preprocess.joblib')

X_train = data['X_train']
X_test = data['X_test']
y_train = data['y_train']
y_test = data['y_test']
preprocess = data['preprocess']


## Cycle 1: Baseline Logistic Regression

In [2]:
base = Pipeline(steps=[
    ('preprocess', preprocess),
    ('model', LogisticRegression(max_iter=2000, random_state=42)),
])

base.fit(X_train, y_train)
print(classification_report(y_test, base.predict(X_test), digits=3))

              precision    recall  f1-score   support

           0      0.912     0.917     0.914      2148
           1      0.909     0.904     0.906      1969

    accuracy                          0.910      4117
   macro avg      0.910     0.910     0.910      4117
weighted avg      0.910     0.910     0.910      4117



## Improve: Tune Regularization (C) + Penalty

We use cross-validation on the training set only. The test set remains untouched until final evaluation.

In [3]:
param_grid = {
    'model__C': [0.1, 0.3, 1.0, 3.0, 10.0],
    'model__penalty': ['l2'],
    'model__solver': ['lbfgs'],
}

pipe = Pipeline(steps=[
    ('preprocess', preprocess),
    ('model', LogisticRegression(max_iter=5000, random_state=42)),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search = GridSearchCV(pipe, param_grid=param_grid, scoring='f1_weighted', cv=cv, n_jobs=-1)
search.fit(X_train, y_train)

print('Best params:', search.best_params_)
print('Best CV score:', search.best_score_)

best_lr = search.best_estimator_
print('Test performance:')
print(classification_report(y_test, best_lr.predict(X_test), digits=3))

Best params: {'model__C': 10.0, 'model__penalty': 'l2', 'model__solver': 'lbfgs'}
Best CV score: 0.9143190631322288
Test performance:
              precision    recall  f1-score   support

           0      0.917     0.916     0.917      2148
           1      0.908     0.910     0.909      1969

    accuracy                          0.913      4117
   macro avg      0.913     0.913     0.913      4117
weighted avg      0.913     0.913     0.913      4117



## Cycle 2: Tune Gradient Boosting

In [4]:
gb = Pipeline(steps=[
    ('preprocess', preprocess),
    ('model', GradientBoostingClassifier(random_state=42)),
])

param_grid = {
    'model__n_estimators': [100, 200],
    'model__learning_rate': [0.05, 0.1],
    'model__max_depth': [2, 3],
}

search = GridSearchCV(gb, param_grid=param_grid, scoring='f1_weighted', cv=cv, n_jobs=-1)
search.fit(X_train, y_train)

print('Best params:', search.best_params_)
print('Best CV score:', search.best_score_)

best_gb = search.best_estimator_
print('Test performance:')
print(classification_report(y_test, best_gb.predict(X_test), digits=3))

Best params: {'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 200}
Best CV score: 0.8865258291840487
Test performance:
              precision    recall  f1-score   support

           0      0.904     0.875     0.889      2148
           1      0.868     0.899     0.883      1969

    accuracy                          0.886      4117
   macro avg      0.886     0.887     0.886      4117
weighted avg      0.887     0.886     0.886      4117



## Notes on Next Improvements

1. Add a minimum attempt filter (e.g. require `FTA * G >= 20`) to reduce noise from tiny samples.
2. Try a time-aware split (train on older seasons, test on newer seasons) to measure generalization across eras.
3. Add domain features:
- `fta_per_min = FTA / MP`
- `usage_proxy = (FGA + 0.44*FTA) / MP`
